In [2]:
%pip install catboost

   ---------------------------------------- 0.0/100.2 MB ? eta -:--:--
   - -------------------------------------- 5.0/100.2 MB 26.1 MB/s eta 0:00:04
   ---- ----------------------------------- 11.5/100.2 MB 30.8 MB/s eta 0:00:03
   ------- -------------------------------- 18.4/100.2 MB 30.3 MB/s eta 0:00:03
   ---------- ----------------------------- 26.2/100.2 MB 31.8 MB/s eta 0:00:03
   ------------ --------------------------- 32.5/100.2 MB 31.4 MB/s eta 0:00:03
   --------------- ------------------------ 38.5/100.2 MB 31.1 MB/s eta 0:00:02
   ---------------- ----------------------- 41.9/100.2 MB 30.9 MB/s eta 0:00:02
   ------------------ --------------------- 46.1/100.2 MB 27.9 MB/s eta 0:00:02
   -------------------- ------------------- 50.6/100.2 MB 26.8 MB/s eta 0:00:02
   ----------------------- ---------------- 57.7/100.2 MB 27.6 MB/s eta 0:00:02
   ------------------------- -------------- 65.0/100.2 MB 28.3 MB/s eta 0:00:02
   ---------------------------- ----------- 72.4/1

In [3]:
# ============================================================
# EXPERIMENT 003
# CatBoost nonlinear model comparison
# ============================================================

from pathlib import Path
from time import time
import warnings

import numpy as np
import pandas as pd

from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold

try:
    from catboost import CatBoostClassifier
except ImportError as exc:
    raise ImportError(
        "CatBoost is not installed. Run `%pip install catboost` "
        "in a notebook cell, restart the kernel if needed, "
        "and rerun this notebook."
    ) from exc


warnings.filterwarnings("ignore")


# ============================================================
# 1. CONFIGURATION
# ============================================================

EXPERIMENT_ID = "EXP-003"

RANDOM_STATE = 42
N_SPLITS = 3

TARGET = "addicted_label"
ID_COLUMN = "id"

XGB_BASELINE_AUC = 0.963034
SUBMISSION_THRESHOLD = 0.9625


# Make paths work when run from either the project root
# or the notebooks directory.
PROJECT_DIR = Path.cwd()

if PROJECT_DIR.name == "notebooks":
    PROJECT_DIR = PROJECT_DIR.parent

DATA_DIR = PROJECT_DIR / "data"
SUBMISSION_DIR = PROJECT_DIR / "submissions"

SUBMISSION_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_PATH = DATA_DIR / "train.csv"
TEST_PATH = DATA_DIR / "test.csv"
SAMPLE_SUBMISSION_PATH = DATA_DIR / "sample_submission.csv"


# ============================================================
# 2. LOAD DATA
# ============================================================

train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)
sample_submission = pd.read_csv(SAMPLE_SUBMISSION_PATH)

print(f"Train shape:             {train.shape}")
print(f"Test shape:              {test.shape}")
print(f"Sample submission shape: {sample_submission.shape}")

assert TARGET in train.columns
assert TARGET not in test.columns
assert len(test) == len(sample_submission)


# ============================================================
# 3. CREATE FEATURES AND TARGET
# ============================================================

X = train.drop(columns=[TARGET, ID_COLUMN]).copy()
y = train[TARGET].astype(int).copy()

X_test = test.drop(columns=[ID_COLUMN]).copy()

assert list(X.columns) == list(X_test.columns)


categorical_columns = X.select_dtypes(
    include=["object", "category", "bool"]
).columns.tolist()

numeric_columns = X.columns.difference(
    categorical_columns
).tolist()


print(f"\nTotal features:       {X.shape[1]}")
print(f"Numerical features:   {len(numeric_columns)}")
print(f"Categorical features: {len(categorical_columns)}")

print("\nCategorical columns:")
print(categorical_columns)


# ============================================================
# 4. PREPARE CATEGORICAL FEATURES
# ============================================================

# CatBoost can natively process missing numerical values.
#
# Categorical values should be consistently represented as
# strings. We replace missing categorical values with an
# explicit category rather than passing None or mixed types.

for column in categorical_columns:
    X[column] = (
        X[column]
        .fillna("__MISSING__")
        .astype(str)
    )

    X_test[column] = (
        X_test[column]
        .fillna("__MISSING__")
        .astype(str)
    )


print("\nRemaining missing values by type:")
print(
    {
        "numeric_train_missing": int(
            X[numeric_columns].isna().sum().sum()
        ),
        "categorical_train_missing": int(
            X[categorical_columns].isna().sum().sum()
        ),
        "numeric_test_missing": int(
            X_test[numeric_columns].isna().sum().sum()
        ),
        "categorical_test_missing": int(
            X_test[categorical_columns].isna().sum().sum()
        )
    }
)


# ============================================================
# 5. MODEL PARAMETERS
# ============================================================

model_parameters = {
    "loss_function": "Logloss",
    "eval_metric": "AUC",

    # Maximum number of boosting rounds.
    "iterations": 2000,

    "learning_rate": 0.05,
    "depth": 8,

    "l2_leaf_reg": 5.0,
    "random_strength": 1.0,

    # Row sampling.
    "bootstrap_type": "Bernoulli",
    "subsample": 0.80,

    # Stop when validation AUC stops improving.
    "early_stopping_rounds": 100,

    "random_seed": RANDOM_STATE,
    "thread_count": -1,

    # Use CPU so the notebook works without GPU setup.
    "task_type": "CPU",

    "allow_writing_files": False,
    "verbose": False
}


# ============================================================
# 6. STRATIFIED CROSS-VALIDATION
# ============================================================

cv = StratifiedKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=RANDOM_STATE
)


out_of_fold_predictions = np.zeros(
    len(train),
    dtype=float
)

test_predictions = np.zeros(
    len(test),
    dtype=float
)

fold_scores = []
best_iterations = []
fold_feature_importances = []

experiment_start = time()


for fold_number, (train_indices, validation_indices) in enumerate(
    cv.split(X, y),
    start=1
):
    fold_start = time()

    X_train_fold = X.iloc[train_indices].copy()
    X_validation_fold = X.iloc[validation_indices].copy()

    y_train_fold = y.iloc[train_indices]
    y_validation_fold = y.iloc[validation_indices]

    model = CatBoostClassifier(
        **model_parameters
    )

    model.fit(
        X_train_fold,
        y_train_fold,
        cat_features=categorical_columns,
        eval_set=(
            X_validation_fold,
            y_validation_fold
        ),
        use_best_model=True,
        verbose=False
    )

    validation_probabilities = model.predict_proba(
        X_validation_fold
    )[:, 1]

    fold_test_probabilities = model.predict_proba(
        X_test
    )[:, 1]

    out_of_fold_predictions[validation_indices] = (
        validation_probabilities
    )

    test_predictions += (
        fold_test_probabilities / N_SPLITS
    )

    fold_auc = roc_auc_score(
        y_validation_fold,
        validation_probabilities
    )

    fold_scores.append(float(fold_auc))

    best_iteration = model.get_best_iteration()
    best_iterations.append(int(best_iteration))

    feature_importance = pd.Series(
        model.get_feature_importance(),
        index=X.columns,
        name=f"fold_{fold_number}"
    )

    fold_feature_importances.append(
        feature_importance
    )

    fold_minutes = (time() - fold_start) / 60

    print(
        f"Fold {fold_number}/{N_SPLITS} | "
        f"AUC: {fold_auc:.6f} | "
        f"Best iteration: {best_iteration} | "
        f"Time: {fold_minutes:.2f} minutes"
    )


# ============================================================
# 7. VALIDATION RESULTS
# ============================================================

overall_oof_auc = roc_auc_score(
    y,
    out_of_fold_predictions
)

mean_fold_auc = float(
    np.mean(fold_scores)
)

std_fold_auc = float(
    np.std(fold_scores)
)

difference_vs_xgb = (
    overall_oof_auc - XGB_BASELINE_AUC
)

total_minutes = (
    time() - experiment_start
) / 60


print("\n" + "=" * 60)
print("EXPERIMENT 003 RESULTS")
print("=" * 60)

for fold_number, score in enumerate(
    fold_scores,
    start=1
):
    print(
        f"Fold {fold_number} AUC: "
        f"{score:.6f}"
    )

print(f"\nMean fold AUC:       {mean_fold_auc:.6f}")
print(f"Fold AUC SD:         {std_fold_auc:.6f}")
print(f"OOF AUC:             {overall_oof_auc:.6f}")
print(f"XGBoost OOF:         {XGB_BASELINE_AUC:.6f}")
print(f"Difference vs XGB:   {difference_vs_xgb:+.6f}")
print(f"Best iterations:     {best_iterations}")
print(f"Runtime:             {total_minutes:.2f} minutes")


# ============================================================
# 8. INTERPRET RESULT
# ============================================================

print("\nInterpretation:")

if overall_oof_auc >= XGB_BASELINE_AUC + 0.001:
    print(
        "CatBoost meaningfully outperformed XGBoost. "
        "It should become the leading individual model."
    )

elif overall_oof_auc >= XGB_BASELINE_AUC - 0.001:
    print(
        "CatBoost and XGBoost are effectively competitive. "
        "Their predictions may be useful for blending."
    )

elif overall_oof_auc >= 0.961:
    print(
        "CatBoost is strong but trails XGBoost. It may still "
        "provide useful prediction diversity for an ensemble."
    )

else:
    print(
        "CatBoost trails XGBoost clearly. XGBoost remains "
        "the primary model family."
    )


# ============================================================
# 9. FEATURE IMPORTANCE
# ============================================================

feature_importance_table = pd.concat(
    fold_feature_importances,
    axis=1
)

feature_importance_table["mean_importance"] = (
    feature_importance_table.mean(axis=1)
)

feature_importance_table["importance_sd"] = (
    feature_importance_table[
        [
            "fold_1",
            "fold_2",
            "fold_3"
        ]
    ].std(axis=1)
)

feature_importance_table = (
    feature_importance_table
    .sort_values(
        "mean_importance",
        ascending=False
    )
)


print("\nMean CatBoost feature importance:")
print(
    feature_importance_table[
        [
            "mean_importance",
            "importance_sd"
        ]
    ].round(4)
)


# ============================================================
# 10. PREDICTION SANITY CHECKS
# ============================================================

prediction_summary = pd.Series(
    test_predictions,
    name="predicted_probability"
).describe()

print("\nTest prediction summary:")
print(prediction_summary)


assert np.isfinite(
    out_of_fold_predictions
).all()

assert np.isfinite(
    test_predictions
).all()

assert (
    (test_predictions >= 0) &
    (test_predictions <= 1)
).all()

assert np.std(test_predictions) > 0


# ============================================================
# 11. CONDITIONAL SUBMISSION
# ============================================================

if overall_oof_auc >= SUBMISSION_THRESHOLD:

    submission = sample_submission.copy()

    assert TARGET in submission.columns

    submission[TARGET] = test_predictions

    submission_path = (
        SUBMISSION_DIR /
        "exp_003_catboost_comparison.csv"
    )

    submission.to_csv(
        submission_path,
        index=False
    )

    print(
        "\nSubmission created because OOF AUC "
        "met the predefined threshold."
    )

    print(f"Saved to:\n{submission_path}")

    print("\nSubmission preview:")
    print(submission.head())

else:
    submission_path = None

    print(
        "\nNo submission created because OOF AUC was below "
        f"{SUBMISSION_THRESHOLD:.4f}."
    )

Train shape:             (691369, 14)
Test shape:              (296302, 13)
Sample submission shape: (296302, 2)

Total features:       12
Numerical features:   9
Categorical features: 3

Categorical columns:
['gender', 'stress_level', 'academic_work_impact']

Remaining missing values by type:
{'numeric_train_missing': 741954, 'categorical_train_missing': 0, 'numeric_test_missing': 317598, 'categorical_test_missing': 0}
Fold 1/3 | AUC: 0.961586 | Best iteration: 1999 | Time: 5.26 minutes
Fold 2/3 | AUC: 0.962336 | Best iteration: 1999 | Time: 5.13 minutes
Fold 3/3 | AUC: 0.962426 | Best iteration: 1998 | Time: 5.24 minutes

EXPERIMENT 003 RESULTS
Fold 1 AUC: 0.961586
Fold 2 AUC: 0.962336
Fold 3 AUC: 0.962426

Mean fold AUC:       0.962116
Fold AUC SD:         0.000376
OOF AUC:             0.962115
XGBoost OOF:         0.963034
Difference vs XGB:   -0.000919
Best iterations:     [1999, 1999, 1998]
Runtime:             15.63 minutes

Interpretation:
CatBoost and XGBoost are effectively c

In [4]:
### Experiment log

In [5]:
from datetime import datetime
from pathlib import Path
import json

import numpy as np
import pandas as pd


EXPERIMENT_LOG_PATH = PROJECT_DIR / "experiment_log.csv"


def log_experiment(
    experiment_id,
    description,
    model,
    features,
    validation_method,
    cv_scores,
    kaggle_score=None,
    changes="",
    submission_file="",
    notes="",
    log_path=EXPERIMENT_LOG_PATH
):
    """
    Add or update one experiment in experiment_log.csv.

    If the experiment_id already exists, its previous row is replaced.
    """

    log_path = Path(log_path)
    log_path.parent.mkdir(parents=True, exist_ok=True)

    cv_scores = [float(score) for score in cv_scores]

    cv_mean = float(np.mean(cv_scores))
    cv_std = float(np.std(cv_scores))

    kaggle_score_value = (
        float(kaggle_score)
        if kaggle_score is not None
        else np.nan
    )

    kaggle_cv_gap = (
        kaggle_score_value - cv_mean
        if pd.notna(kaggle_score_value)
        else np.nan
    )

    experiment_record = {
        "experiment_id": experiment_id,
        "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "description": description,
        "model": model,
        "features": json.dumps(list(features)),
        "n_features": len(features),
        "validation_method": validation_method,
        "cv_scores": json.dumps(cv_scores),
        "cv_mean": cv_mean,
        "cv_std": cv_std,
        "kaggle_score": kaggle_score_value,
        "kaggle_cv_gap": kaggle_cv_gap,
        "changes": changes,
        "submission_file": submission_file,
        "notes": notes
    }

    if log_path.exists():
        experiments = pd.read_csv(log_path)

        # Prevent duplicate rows when rerunning the same experiment cell.
        if "experiment_id" in experiments.columns:
            experiments = experiments[
                experiments["experiment_id"] != experiment_id
            ].copy()
    else:
        experiments = pd.DataFrame()

    new_row = pd.DataFrame([experiment_record])

    experiments = pd.concat(
        [experiments, new_row],
        ignore_index=True
    )

    experiments = experiments.sort_values(
        by="experiment_id"
    ).reset_index(drop=True)

    experiments.to_csv(log_path, index=False)

    print(f"Logged {experiment_id}")
    print(f"CV mean:       {cv_mean:.6f}")
    print(f"CV SD:         {cv_std:.6f}")

    if pd.notna(kaggle_score_value):
        print(f"Kaggle score:  {kaggle_score_value:.6f}")
        print(f"Kaggle-CV gap: {kaggle_cv_gap:+.6f}")

    print(f"Log saved to:  {log_path}")

    return experiments

In [6]:
experiments = log_experiment(
    experiment_id="EXP-003",
    description=(
        "CatBoost comparison using native numerical missing-value handling "
        "and explicit categorical missing-value categories."
    ),
    model="CatBoostClassifier",
    features=X.columns.tolist(),
    validation_method="3-fold StratifiedKFold with ROC AUC",
    cv_scores=[
        0.961586,
        0.962336,
        0.962426
    ],
    kaggle_score=None,
    changes=(
        "Replaced XGBoost and one-hot preprocessing with CatBoost's native "
        "categorical-feature processing and missing-value handling."
    ),
    submission_file="",
    notes=(
        "OOF AUC was 0.962115 with fold SD 0.000376. CatBoost trailed "
        "XGBoost by only 0.000919, making the models effectively competitive. "
        "Runtime was 15.63 minutes versus approximately 2.88 minutes for "
        "XGBoost. Best iterations were 1999, 1999, and 1998, indicating that "
        "the iteration limit was reached. No submission was created because "
        "OOF AUC was below the predefined 0.9625 threshold."
    )
)

experiments.tail()

Logged EXP-003
CV mean:       0.962116
CV SD:         0.000377
Log saved to:  C:\Users\Owner\Documents\Github\machine-learning-lab\00-Kaggle\02-smartphone-addiction\experiment_log.csv


,experiment_id,timestamp,description,model,features,n_features,validation_method,cv_scores,cv_mean,cv_std,kaggle_score,kaggle_cv_gap,changes,submission_file,notes
0,EXP-001,2026-08-02 21:49:55,Initial XGBoost baseline using median-imputed ...,XGBClassifier,"[""age"", ""daily_screen_time_hours"", ""social_med...",12,3-fold StratifiedKFold with ROC AUC,"[0.962477, 0.963382, 0.963244]",0.963034,0.000398,0.96449,0.001456,Established the first end-to-end baseline usin...,exp_001_xgb_baseline.csv,OOF AUC was 0.963034 with fold SD 0.000398. Ka...
1,EXP-002,2026-08-02 21:53:06,Logistic-regression control using standardized...,LogisticRegression,"[""age"", ""daily_screen_time_hours"", ""social_med...",12,3-fold StratifiedKFold with ROC AUC,"[0.910347, 0.911866, 0.912106]",0.911440,0.000779,NaN,NaN,Replaced the XGBoost baseline with a regulariz...,NaN,OOF AUC was 0.911437 with fold SD 0.000779. Th...
2,EXP-003,2026-08-02 22:11:48,CatBoost comparison using native numerical mis...,CatBoostClassifier,"[""age"", ""daily_screen_time_hours"", ""social_med...",12,3-fold StratifiedKFold with ROC AUC,"[0.961586, 0.962336, 0.962426]",0.962116,0.000377,NaN,NaN,Replaced XGBoost and one-hot preprocessing wit...,,OOF AUC was 0.962115 with fold SD 0.000376. Ca...
